In [ ]:
import os
for variable in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = "1"
from pathlib import Path
RUNTIME_ROOT = Path(os.environ.get("SAFETY_WATCH_RUNTIME_ROOT", "/runtime"))
PROJECT_ROOT = Path(os.environ.get("SAFETY_WATCH_PROJECT_ROOT", "/workspace"))
SEED = 42
VALIDATION_FRACTION = 0.20
TARGET_FPR = 0.05
RUN_MODEL_SELECTION = True
CV_FOLDS = 3
SEARCH_C_VALUES = (0.3, 1.0, 3.0, 10.0, 30.0, 100.0)
FREEZE_MODEL = False
RUN_FINAL_TEST = False
MODEL_CONFIG = {
    "analyzer": "char", "ngram_range": (3, 5), "lowercase": True,
    "min_df": 2, "max_features": 100_000, "sublinear_tf": True, "norm": "l2",
    "solver": "liblinear", "C": 1.0, "class_weight": None, "max_iter": 1000,
}
# sklearn 1.9: l1_ratio=0.0 specifies L2 without the deprecated penalty parameter.
DATA_ROOT = RUNTIME_ROOT / "data"
MODEL_ROOT = RUNTIME_ROOT / "models"
RESULT_ROOT = RUNTIME_ROOT / "results"
LOG_ROOT = RUNTIME_ROOT / "logs"


In [ ]:
import sys, platform, json, hashlib, datetime, uuid, warnings, time, resource, shutil
import numpy as np
import pandas as pd
import scipy, sklearn, joblib, matplotlib
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, average_precision_score

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

require(sys.version_info[:2] == (3, 12), "Use the container Python 3.12 interpreter.")
require(platform.machine() == "s390x", "Training must run natively on LinuxONE s390x.")
require(sklearn.__version__ == "1.9.0", "Use the pinned scikit-learn 1.9.0 image.")
require((PROJECT_ROOT / "AGENTS.md").is_file(), "Mount repository as PROJECT_ROOT.")
require(DATA_ROOT.is_dir(), "Mount existing VM data read-only.")
environment = {
    "python": platform.python_version(), "architecture": platform.machine(),
    "interpreter": sys.executable, "sklearn": sklearn.__version__, "numpy": np.__version__,
    "scipy": scipy.__version__, "pandas": pd.__version__, "joblib": joblib.__version__,
    "matplotlib": matplotlib.__version__,
}
def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True, allow_nan=False)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n")
    temporary.replace(path)

run_timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat()
run_id = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
run_dir = RESULT_ROOT / "guardrail" / run_id
run_dir.mkdir(parents=True, exist_ok=False)
evidence_path = LOG_ROOT / "classifier" / (run_id + ".jsonl")
evidence_path.parent.mkdir(parents=True, exist_ok=True)

def evidence(event, payload):
    previous = "0" * 64
    if evidence_path.exists():
        previous = json.loads(evidence_path.read_text().splitlines()[-1])["hash"]
    record = {"previous_hash": previous, "event": event, "payload": payload}
    record["hash"] = hashlib.sha256(canonical(record).encode()).hexdigest()
    with evidence_path.open("a") as handle:
        handle.write(canonical(record) + "\n")
    return record["hash"]

def verify_evidence(path):
    previous, count = "0" * 64, 0
    for line in Path(path).read_text().splitlines():
        record = json.loads(line)
        stored = record.pop("hash")
        require(record["previous_hash"] == previous, "Evidence link verification failed.")
        require(hashlib.sha256(canonical(record).encode()).hexdigest() == stored, "Evidence digest verification failed.")
        previous, count = stored, count + 1
    return {"records": count, "head": previous}

source_digest = sha256_file(PROJECT_ROOT / "notebooks" / "01_initial_guardrail.ipynb")
configuration = {
    "seed": SEED, "validation_fraction": VALIDATION_FRACTION, "target_fpr": TARGET_FPR,
    "model": MODEL_CONFIG, "target": "toxicity OR jailbreaking",
    "deduplication": "exact text; exclude train/test overlaps; prefer human annotation then stable ID",
}
evidence("environment", {"environment": environment, "configuration": configuration, "notebook_sha256": source_digest})
print(json.dumps(environment, indent=2))
print("Development run:", run_id)

def choose_threshold(benign, maximum_fpr):
    benign = np.asarray(benign, dtype=float)
    require(benign.size > 0 and np.isfinite(benign).all(), "Threshold needs finite benign scores.")
    candidates = np.unique(np.concatenate(([0.0], benign, np.nextafter(benign, np.inf), [np.nextafter(1.0, np.inf)])))
    allowed = int(np.floor(maximum_fpr * benign.size + 1e-12))
    for candidate in candidates:
        if int(np.count_nonzero(benign >= candidate)) <= allowed:
            return float(candidate)
    raise RuntimeError("No threshold satisfies the FPR target.")



In [ ]:
manifest = json.loads((DATA_ROOT / "download-manifest.json").read_text())
entries = [entry for entry in manifest if entry.get("repo") == "lmsys/toxic-chat"]
require(len(entries) == 1, "Expected one ToxicChat manifest entry.")
dataset_entry = entries[0]
raw_root = DATA_ROOT / "raw" / "toxicchat0124"
def manifest_file(filename):
    matches = [item for item in dataset_entry["files"] if item["path"] == "data/0124/" + filename]
    require(len(matches) == 1, "Expected unique ToxicChat 0124 file.")
    item = matches[0]
    path = raw_root / item["path"]
    require(path.is_file(), "ToxicChat file missing.")
    digest = sha256_file(path)
    require(digest == item["sha256"], "Dataset SHA-256 mismatch; do not train.")
    require(path.stat().st_size == item["bytes"], "Dataset byte count mismatch.")
    return path, digest

train_path, train_digest = manifest_file("toxic-chat_annotation_train.csv")
test_path, test_digest = manifest_file("toxic-chat_annotation_test.csv")
required_columns = ["conv_id", "user_input", "human_annotation", "toxicity", "jailbreaking"]
train = pd.read_csv(train_path, usecols=required_columns, dtype={"conv_id": "string", "user_input": "string"})
require(not train[required_columns].isna().any().any(), "Missing required training fields.")
require(train["user_input"].str.strip().str.len().gt(0).all(), "Empty training prompts.")
for label in ("toxicity", "jailbreaking"):
    require(train[label].isin([0, 1]).all(), "Labels must be binary 0/1.")
train["human_annotation"] = train["human_annotation"].astype(str).map({"True": True, "False": False})
require(train["human_annotation"].notna().all(), "Invalid annotation flag.")
train["unsafe"] = (train["toxicity"].eq(1) | train["jailbreaking"].eq(1)).astype(int)
train["prompt_id"] = ["toxicchat0124:train:" + str(index) for index in range(len(train))]
train["text_sha256"] = train["user_input"].map(lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest())
dataset_metadata = {
    "repo": "lmsys/toxic-chat", "revision": dataset_entry["revision"],
    "train_sha256": train_digest, "test_sha256": test_digest,
    "official_training_rows": len(train), "license": "CC-BY-NC-4.0",
}
evidence("data_verified", dataset_metadata)
print("Official training rows:", len(train))
print(train.groupby(["human_annotation", "unsafe"]).size().rename("records").to_string())


In [ ]:
test_text = pd.read_csv(test_path, usecols=["user_input"], dtype={"user_input": "string"})
require(test_text["user_input"].notna().all(), "Missing test text.")
test_hashes = set(test_text["user_input"].map(lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest()))
require(train.groupby("text_sha256")["unsafe"].nunique().le(1).all(), "Conflicting duplicate training targets.")
overlap = train["text_sha256"].isin(test_hashes)
audit_columns = ["prompt_id", "conv_id", "text_sha256", "human_annotation", "unsafe"]
excluded_overlap = train.loc[overlap, audit_columns].copy()
excluded_overlap["reason"] = "exact_test_text_overlap"
eligible = train.loc[~overlap].sort_values(
    ["human_annotation", "conv_id", "prompt_id"], ascending=[False, True, True], kind="stable"
).copy()
duplicate = eligible.duplicated("text_sha256", keep="first")
excluded_duplicate = eligible.loc[duplicate, audit_columns].copy()
excluded_duplicate["reason"] = "duplicate_training_text"
excluded = pd.concat([excluded_overlap, excluded_duplicate], ignore_index=True)
eligible = eligible.loc[~duplicate].sort_values("prompt_id", kind="stable").reset_index(drop=True)
strata = eligible["human_annotation"].astype(str) + ":" + eligible["unsafe"].astype(str)
require(strata.value_counts().ge(2).all(), "Split stratum has fewer than two records.")
fit_idx, validation_idx = train_test_split(
    np.arange(len(eligible)), test_size=VALIDATION_FRACTION, random_state=SEED, stratify=strata,
)
fit, validation = eligible.iloc[fit_idx].copy(), eligible.iloc[validation_idx].copy()
require(set(fit["prompt_id"]).isdisjoint(validation["prompt_id"]), "Fit/validation IDs overlap.")
require(set(fit["text_sha256"]).isdisjoint(validation["text_sha256"]), "Fit/validation text overlaps.")
require(set(eligible["text_sha256"]).isdisjoint(test_hashes), "Eligible training overlaps test.")
require(fit["unsafe"].nunique() == 2 and validation["unsafe"].nunique() == 2, "Each partition needs both classes.")
partitions = {
    "fit": fit[audit_columns].to_dict(orient="records"),
    "validation": validation[audit_columns].to_dict(orient="records"),
    "excluded": excluded.to_dict(orient="records"),
}
write_json(run_dir / "partitions.json", partitions)
partition_digest = sha256_file(run_dir / "partitions.json")
split_summary = {
    "cross_boundary_excluded": int(overlap.sum()), "within_training_excluded": int(duplicate.sum()),
    "eligible": len(eligible), "fit": len(fit), "validation": len(validation),
    "partitions_sha256": partition_digest,
}
evidence("partitions_locked", split_summary)
print(json.dumps(split_summary, indent=2))
del test_text, test_hashes


In [ ]:
selection_metadata = {"enabled": RUN_MODEL_SELECTION, "scope": "fitting partition only"}
if RUN_MODEL_SELECTION:
    search_started = time.perf_counter()
    candidates = [{"C": float(value), "class_weight": weight}
                  for value in SEARCH_C_VALUES for weight in (None, "balanced")]
    fold_results = [[] for _ in candidates]
    fitting_strata = fit["human_annotation"].astype(str) + ":" + fit["unsafe"].astype(str)
    splitter = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    fold_manifest = []
    for fold_number, (inner_fit_idx, inner_check_idx) in enumerate(
        splitter.split(np.zeros(len(fit)), fitting_strata), start=1
    ):
        inner_fit, inner_check = fit.iloc[inner_fit_idx], fit.iloc[inner_check_idx]
        require(set(inner_fit["prompt_id"]).isdisjoint(inner_check["prompt_id"]), "CV IDs overlap.")
        require(set(inner_fit["text_sha256"]).isdisjoint(inner_check["text_sha256"]), "CV text overlaps.")
        require(set(inner_fit["prompt_id"]).isdisjoint(validation["prompt_id"]), "Outer validation leaked into CV.")
        fold_manifest.append({
            "fold": fold_number, "fit_ids": inner_fit["prompt_id"].tolist(),
            "check_ids": inner_check["prompt_id"].tolist(),
        })
        vectorizer = TfidfVectorizer(
            analyzer=MODEL_CONFIG["analyzer"], ngram_range=MODEL_CONFIG["ngram_range"],
            lowercase=MODEL_CONFIG["lowercase"], min_df=MODEL_CONFIG["min_df"],
            max_features=MODEL_CONFIG["max_features"], sublinear_tf=MODEL_CONFIG["sublinear_tf"],
            norm=MODEL_CONFIG["norm"],
        )
        # Fit vocabulary/IDF separately within each inner training fold.
        inner_x = vectorizer.fit_transform(inner_fit["user_input"])
        check_x = vectorizer.transform(inner_check["user_input"])
        check_human = inner_check["human_annotation"].to_numpy(dtype=bool)
        check_y = inner_check["unsafe"].to_numpy(dtype=int)
        require(np.unique(check_y[check_human]).size == 2, "Human CV fold needs both classes.")
        for index, candidate in enumerate(candidates):
            classifier = LogisticRegression(
                solver="liblinear", l1_ratio=0.0, C=candidate["C"],
                class_weight=candidate["class_weight"], max_iter=MODEL_CONFIG["max_iter"],
                random_state=SEED,
            )
            started = time.perf_counter()
            with warnings.catch_warnings():
                warnings.simplefilter("error", ConvergenceWarning)
                classifier.fit(inner_x, inner_fit["unsafe"])
            unsafe_index = int(np.flatnonzero(classifier.classes_ == 1)[0])
            scores = classifier.predict_proba(check_x)[:, unsafe_index]
            benign = scores[check_human & (check_y == 0)]
            cutoff = choose_threshold(benign, TARGET_FPR)
            unsafe = check_human & (check_y == 1)
            tp = int(np.count_nonzero(scores[unsafe] >= cutoff))
            fp = int(np.count_nonzero(benign >= cutoff))
            fold_results[index].append({
                "fold": fold_number, "tp": tp, "unsafe": int(unsafe.sum()),
                "fp": fp, "benign": len(benign), "threshold": cutoff,
                "average_precision": float(average_precision_score(check_y[check_human], scores[check_human])),
                "fit_seconds": time.perf_counter() - started,
            })
        print("Completed fitting-only CV fold", fold_number, "of", CV_FOLDS)
        del inner_x, check_x, classifier, vectorizer

    leaderboard = []
    for candidate, results in zip(candidates, fold_results):
        leaderboard.append({
            **candidate,
            "pooled_human_recall": sum(r["tp"] for r in results) / sum(r["unsafe"] for r in results),
            "pooled_human_fpr": sum(r["fp"] for r in results) / sum(r["benign"] for r in results),
            "mean_average_precision": float(np.mean([r["average_precision"] for r in results])),
            "folds": results,
        })
    # Predeclared ranking: recall at the FPR constraint, then AP, then smaller C, then no weighting.
    chosen = max(leaderboard, key=lambda row: (
        row["pooled_human_recall"], row["mean_average_precision"],
        -row["C"], row["class_weight"] is None,
    ))
    MODEL_CONFIG.update({key: chosen[key] for key in ("C", "class_weight")})
    selection_metadata = {
        "enabled": True, "scope": "fitting partition only", "folds": CV_FOLDS,
        "candidate_count": len(candidates), "classifier_fits": len(candidates) * CV_FOLDS,
        "ranking": "pooled human recall at <=5% fold FPR, then mean AP, smaller C, no weighting",
        "selected_parameters": {key: chosen[key] for key in ("C", "class_weight")},
        "selected_cv_recall": chosen["pooled_human_recall"],
        "selected_cv_average_precision": chosen["mean_average_precision"],
        "search_seconds": time.perf_counter() - search_started,
    }
    write_json(run_dir / "model-selection.json", {
        "selection": selection_metadata, "leaderboard": leaderboard, "fold_membership": fold_manifest,
    })
    selection_metadata["search_artifact_sha256"] = sha256_file(run_dir / "model-selection.json")
    table = pd.DataFrame([{key: row[key] for key in
             ("C", "class_weight", "pooled_human_recall", "pooled_human_fpr", "mean_average_precision")}
             for row in leaderboard]).sort_values(
                 ["pooled_human_recall", "mean_average_precision"], ascending=False
             )
    print(table.to_string(index=False))
    print("Selected settings:", json.dumps(selection_metadata["selected_parameters"]))
else:
    print("Model selection disabled; original baseline settings.")
configuration["model_selection"] = selection_metadata
evidence("model_selection_completed", selection_metadata)


In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer=MODEL_CONFIG["analyzer"], ngram_range=MODEL_CONFIG["ngram_range"],
        lowercase=MODEL_CONFIG["lowercase"], min_df=MODEL_CONFIG["min_df"],
        max_features=MODEL_CONFIG["max_features"], sublinear_tf=MODEL_CONFIG["sublinear_tf"],
        norm=MODEL_CONFIG["norm"],
    )),
    ("classifier", LogisticRegression(
        solver=MODEL_CONFIG["solver"], l1_ratio=0.0, C=MODEL_CONFIG["C"],
        class_weight=MODEL_CONFIG["class_weight"], max_iter=MODEL_CONFIG["max_iter"], random_state=SEED,
    )),
])
started = time.perf_counter()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    pipeline.fit(fit["user_input"], fit["unsafe"])
training_seconds = time.perf_counter() - started
require(not any(issubclass(item.category, ConvergenceWarning) for item in caught), "Classifier did not converge.")
unsafe_column = int(np.flatnonzero(pipeline.classes_ == 1)[0])
started = time.perf_counter()
validation_scores = pipeline.predict_proba(validation["user_input"])[:, unsafe_column]
scoring_seconds = time.perf_counter() - started
require(np.isfinite(validation_scores).all(), "Non-finite validation scores.")
require(((validation_scores >= 0) & (validation_scores <= 1)).all(), "Scores outside [0, 1].")
feature_count = len(pipeline.named_steps["tfidf"].vocabulary_)
fit_summary = {
    "training_seconds": training_seconds, "validation_scoring_seconds": scoring_seconds,
    "validation_records": len(validation), "feature_count": feature_count,
    "solver_iterations": pipeline.named_steps["classifier"].n_iter_.tolist(),
    "process_peak_rss_mib": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024,
}
evidence("model_fitted", fit_summary)
print(json.dumps(fit_summary, indent=2))


In [ ]:
human_mask = validation["human_annotation"].to_numpy(dtype=bool)
labels = validation["unsafe"].to_numpy(dtype=int)
benign_scores = validation_scores[human_mask & (labels == 0)]
require(len(benign_scores) > 0, "No human-annotated benign validation.")
require(np.unique(labels[human_mask]).size == 2, "Human validation needs both classes.")


threshold = choose_threshold(benign_scores, TARGET_FPR)
predictions = (validation_scores >= threshold).astype(int)
threshold_summary = {
    "threshold": threshold, "decision": "score >= threshold", "target_fpr": TARGET_FPR,
    "benign_human_validation_records": len(benign_scores),
    "benign_human_refused": int(np.count_nonzero(benign_scores >= threshold)),
    "observed_fpr": float(np.mean(benign_scores >= threshold)),
}
require(threshold_summary["observed_fpr"] <= TARGET_FPR + 1e-12, "Threshold exceeds FPR target.")
evidence("threshold_selected", threshold_summary)
print(json.dumps(threshold_summary, indent=2))


In [ ]:
def metrics(y, scores, decision_threshold):
    y, scores = np.asarray(y, dtype=int), np.asarray(scores, dtype=float)
    require(len(y) == len(scores) and len(y) > 0, "Invalid metric sizes.")
    require(np.unique(y).size == 2, "Metrics require both classes.")
    require(np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all(), "Invalid metric scores.")
    predicted = (scores >= decision_threshold).astype(int)
    tn, fp, fn, tp = (int(value) for value in confusion_matrix(y, predicted, labels=[0, 1]).ravel())
    return {
        "records": len(y), "benign": tn + fp, "unsafe": tp + fn,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "precision": float(precision_score(y, predicted, zero_division=0)),
        "recall": float(recall_score(y, predicted, zero_division=0)),
        "f1": float(f1_score(y, predicted, zero_division=0)),
        "fpr": fp / (tn + fp), "false_negative_rate": fn / (tp + fn),
        "average_precision": float(average_precision_score(y, scores)),
    }

validation_metrics = {
    "human_annotated": metrics(labels[human_mask], validation_scores[human_mask], threshold),
    "full_validation_supplementary": metrics(labels, validation_scores, threshold),
}
print(pd.DataFrame(validation_metrics).T.to_string())
subgroups = {}
for source_label in ("toxicity", "jailbreaking"):
    mask = human_mask & validation[source_label].eq(1).to_numpy()
    subgroups[source_label] = {
        "records": int(mask.sum()), "recall": float(predictions[mask].mean()) if mask.any() else None,
    }
print("Human unsafe subgroups (may overlap):", json.dumps(subgroups))
write_json(run_dir / "validation-metrics.json", {"metrics": validation_metrics, "unsafe_subgroups": subgroups})
write_json(run_dir / "threshold.json", threshold_summary)
evidence("validation_evaluated", {"metrics": validation_metrics, "unsafe_subgroups": subgroups})
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for value, name in ((0, "Benign"), (1, "Unsafe")):
    mask = human_mask & (labels == value)
    axes[0].hist(validation_scores[mask], bins=np.linspace(0, 1, 21), alpha=0.6, label=name)
axes[0].axvline(threshold, color="black", linestyle="--", label="Refusal threshold")
axes[0].set(xlabel="Unsafe score", ylabel="Records", title="Human-annotated validation scores")
axes[0].legend()
m = validation_metrics["human_annotated"]
axes[1].imshow([[m["tn"], m["fp"]], [m["fn"], m["tp"]]], cmap="Blues")
for row, values in enumerate(((m["tn"], m["fp"]), (m["fn"], m["tp"]))):
    for column, value in enumerate(values):
        axes[1].text(column, row, str(value), ha="center", va="center",
                     color="white" if value > max(m["tn"], m["fp"], m["fn"], m["tp"]) / 2 else "black")
axes[1].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Allow", "Refuse"],
            yticklabels=["Benign", "Unsafe"], xlabel="Decision", ylabel="Label", title="Validation confusion counts")
fig.tight_layout()
fig.savefig(run_dir / "validation-summary.png", dpi=150)
plt.show()
plt.close(fig)


In [ ]:
development_model = run_dir / "guardrail.development.joblib"
joblib.dump(pipeline, development_model)
model_digest = sha256_file(development_model)
model_card = {
    "name": "initial-guardrail", "status": "development",
    "evidence_label": "verified",
    "verification_scope": "development validation on native s390x VM",
    "environment": environment, "dataset": dataset_metadata, "configuration": configuration,
    "split_summary": split_summary, "model_selection": selection_metadata, "threshold": threshold_summary, "validation_metrics": validation_metrics,
    "runtime": {**fit_summary, "model_bytes": development_model.stat().st_size},
    "model_sha256": model_digest, "notebook_sha256": source_digest, "run_id": run_id,
    "score_semantics": "estimated binary unsafe probability; not validated harm severity",
    "limitations": [
        "Public-data development baseline; no production claim.",
        "Lexical features can miss contextual, multilingual, or novel attacks.",
        "No canary or error-correlation measurement yet.",
        "Validation FPR does not guarantee future FPR.",
        "Final test evaluation has not been performed.",
    ],
}
write_json(run_dir / "model-card.json", model_card)
score_path = run_dir / "validation-scores.jsonl"
with score_path.open("x") as handle:
    for prompt_id, score in zip(validation["prompt_id"], validation_scores):
        record = {
            "prompt_id": prompt_id, "guardrail_score": float(score),
            "decision": "refuse" if score >= threshold else "allow",
            "threshold": threshold, "model_sha256": model_digest,
            "timestamp": run_timestamp, "split": "validation",
        }
        handle.write(canonical(record) + chr(10))
evidence("validation_scores_exported", {
    "records": len(validation), "scores_sha256": sha256_file(score_path),
    "model_sha256": model_digest,
})
reloaded = joblib.load(development_model)
probe = validation["user_input"].iloc[:100]
require(len(probe) == 100, "Need 100 validation records for reload verification.")
expected = pipeline.predict_proba(probe)[:, unsafe_column]
actual = reloaded.predict_proba(probe)[:, unsafe_column]
np.testing.assert_allclose(actual, expected, rtol=0, atol=1e-12)
reproduction = sklearn.base.clone(pipeline)
with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    reproduction.fit(fit["user_input"], fit["unsafe"])
repeat_scores = reproduction.predict_proba(validation["user_input"])[:, unsafe_column]
np.testing.assert_allclose(repeat_scores, validation_scores, rtol=0, atol=1e-12)
require(choose_threshold(repeat_scores[human_mask & (labels == 0)], TARGET_FPR) == threshold, "Repeated fit changed threshold.")
evidence("artifact_reload_and_reproduction_verified", {
    "model_sha256": model_digest, "reload_records": len(probe),
    "reproduced_records": len(validation), "score_absolute_tolerance": 1e-12,
})
del reproduction
print("Reload and independent repeated-fit checks passed.")
print("Development model bytes:", development_model.stat().st_size)
print("Model SHA-256:", model_digest)


In [ ]:
again_fit, again_validation = train_test_split(
    np.arange(len(eligible)), test_size=VALIDATION_FRACTION, random_state=SEED, stratify=strata,
)
require(np.array_equal(again_fit, fit_idx) and np.array_equal(again_validation, validation_idx),
        "Repeated split changed membership.")
tied_real_scores = np.repeat(benign_scores, 2)
tied_threshold = choose_threshold(tied_real_scores, TARGET_FPR)
require(np.mean(tied_real_scores >= tied_threshold) <= TARGET_FPR + 1e-12, "Tie handling failed.")
verify_evidence(evidence_path)
tamper_path = run_dir / "evidence-tamper-check.jsonl"
records = evidence_path.read_text().splitlines()
tampered = json.loads(records[0])
tampered["payload"]["tamper_check"] = True
records[0] = canonical(tampered)
tamper_path.write_text("\n".join(records) + "\n")
detected = False
try:
    verify_evidence(tamper_path)
except RuntimeError:
    detected = True
finally:
    tamper_path.unlink()
require(detected, "Modified evidence was not detected.")
evidence("development_checks_passed", {
    "partition_reproducibility": True, "threshold_ties": True, "serialization": True,
    "independent_fit_reproducibility": True, "evidence_tampering_detected": detected,
})
print("All development checks passed.")
print("Evidence:", json.dumps(verify_evidence(evidence_path)))


In [ ]:
frozen_path = MODEL_ROOT / "guardrail.joblib"
cards_path = MODEL_ROOT / "model-cards.json"
if FREEZE_MODEL:
    MODEL_ROOT.mkdir(parents=True, exist_ok=True)
    require(not frozen_path.exists(), "Frozen guardrail exists; do not overwrite.")
    cards = json.loads(cards_path.read_text()) if cards_path.exists() else {}
    require(isinstance(cards, dict) and "guardrail" not in cards, "Existing cards need explicit reconciliation.")
    frozen_card = {**model_card, "status": "frozen"}
    with frozen_path.open("xb") as destination, development_model.open("rb") as source:
        shutil.copyfileobj(source, destination)
    require(sha256_file(frozen_path) == model_digest, "Frozen model copy mismatch.")
    cards["guardrail"] = frozen_card
    write_json(cards_path, cards)
    evidence("guardrail_frozen", {"model_sha256": model_digest, "threshold": threshold})
    print("Frozen guardrail saved without refitting.")
else:
    evidence("freeze_skipped", {"reason": "FREEZE_MODEL is false"})
    print("Freeze disabled; development artifacts only.")


In [ ]:
final_receipt = RESULT_ROOT / "guardrail-final-evaluation.json"
final_claim = RESULT_ROOT / "guardrail-final-evaluation.started.json"
if RUN_FINAL_TEST:
    require(frozen_path.is_file() and cards_path.is_file(), "Freeze before final evaluation.")
    frozen_digest = sha256_file(frozen_path)
    frozen_card = json.loads(cards_path.read_text())["guardrail"]
    require(frozen_card["model_sha256"] == frozen_digest, "Frozen model/card digest mismatch.")
    require(frozen_card["status"] == "frozen", "Model card is not frozen.")
    require(frozen_card["dataset"]["train_sha256"] == train_digest, "Frozen card references different training data.")
    require(frozen_card["dataset"]["test_sha256"] == test_digest, "Frozen card references different test data.")
    final_threshold = frozen_card["threshold"]["threshold"]
    if final_receipt.exists():
        saved_final = json.loads(final_receipt.read_text())
        require(saved_final["model_sha256"] == frozen_digest and saved_final["test_sha256"] == test_digest and saved_final["threshold"] == final_threshold,
                "Final test already used with different artifacts; no scoring allowed.")
        print("Saved final metrics:", json.dumps(saved_final["metrics"], indent=2))
    else:
        require(not final_claim.exists(), "Final evaluation already started; inspect evidence before recovery.")
        claim = {"model_sha256": frozen_digest, "test_sha256": test_digest, "run_id": run_id}
        with final_claim.open("x") as handle:
            handle.write(canonical(claim) + "\n")
        evidence("final_evaluation_started", claim)
        final_model = joblib.load(frozen_path)
        final_data = pd.read_csv(test_path, usecols=required_columns,
                                 dtype={"conv_id": "string", "user_input": "string"})
        require(not final_data[required_columns].isna().any().any(), "Missing final test fields.")
        require(final_data["user_input"].str.strip().str.len().gt(0).all(), "Empty final test prompts.")
        for label in ("toxicity", "jailbreaking"):
            require(final_data[label].isin([0, 1]).all(), "Invalid final test labels.")
        final_human = final_data["human_annotation"].astype(str).map({"True": True, "False": False})
        require(final_human.notna().all(), "Invalid final annotation flags.")
        final_labels = (final_data["toxicity"].eq(1) | final_data["jailbreaking"].eq(1)).to_numpy(dtype=int)
        final_column = int(np.flatnonzero(final_model.classes_ == 1)[0])
        final_scores = final_model.predict_proba(final_data["user_input"])[:, final_column]
        final_mask = final_human.to_numpy(dtype=bool)
        final_metrics = {
            "human_annotated": metrics(final_labels[final_mask], final_scores[final_mask], final_threshold),
            "full_test_supplementary": metrics(final_labels, final_scores, final_threshold),
        }
        receipt = {**claim, "threshold": final_threshold, "metrics": final_metrics}
        write_json(final_receipt, receipt)
        evidence("final_evaluation_completed", receipt)
        print("Final metrics:", json.dumps(final_metrics, indent=2))
else:
    evidence("final_evaluation_skipped", {"reason": "RUN_FINAL_TEST is false"})
    print("Official test scoring remains disabled.")


In [ ]:
summary = {
    "run_id": run_id, "results_directory": str(run_dir), "evidence_file": str(evidence_path),
    "model_sha256": model_digest, "partition_sha256": partition_digest, "threshold": threshold,
    "human_validation": validation_metrics["human_annotated"],
    "freeze_requested": FREEZE_MODEL, "final_test_requested": RUN_FINAL_TEST,
    "evidence": verify_evidence(evidence_path),
}
write_json(run_dir / "run-summary.json", summary)
print(json.dumps(summary, indent=2))
